In [4]:
# Install dependencies
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q ftfy regex tqdm h5py pillow pyyaml einops
!pip install -q git+https://github.com/openai/CLIP


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
!pip install torch-fidelity lpips ftfy regex tqdm

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
import os, sys, glob

CANDIDATES = [
    "/content/drive/MyDrive/DCGAN-Text-to-Image"
]
DRIVE_PROJ = None
for p in CANDIDATES:
    if os.path.exists(p) and os.path.isdir(p):
        DRIVE_PROJ = p
        break
assert DRIVE_PROJ, "Project folder not found on Drive."

%cd $DRIVE_PROJ
!ls -la


/content/drive/MyDrive/DCGAN-Text-to-Image
total 1966355
drwx------ 2 root root       4096 Nov 13 11:17 102flowers
drwx------ 2 root root       4096 Nov 13 01:10 cfg
-rw------- 1 root root       3081 Nov 13 01:31 class_names.py
-rw------- 1 root root 2013486599 Nov 13 01:20 flowers.hdf5
drwx------ 2 root root       4096 Nov 13 01:25 .ipynb_checkpoints
drwx------ 2 root root       4096 Nov 13 01:25 models
drwx------ 2 root root       4096 Nov 13 01:41 output_clip
drwx------ 2 root root       4096 Nov 13 01:41 __pycache__
-rw------- 1 root root       2119 Nov 13 01:29 runtime.py
-rw------- 1 root root       2366 Nov 13 01:29 select_best_from_prompt.py
-rw------- 1 root root       5112 Nov 13 11:51 select_best_gen.py
drwx------ 2 root root       4096 Nov 13 05:42 tools
-rw------- 1 root root      16114 Nov 13 01:47 trainer.py
-rw------- 1 root root       1458 Nov 13 01:28 txt2image_dataset.py


In [3]:
import os, random, numpy as np, torch
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except:
    pass


In [ ]:
!python runtime.py --cfg cfg/flowers_clip.yml


Using device: cuda
100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 114MiB/s]
Epoch 1/200:   0%|                                                          | 0/459 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:829: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch 9/200: 100%|█| 459/459 [01:23<00:00,  5.49it/s, D=0.528, G_adv=3.574, CLIP=0.777, G_tot=3.714]
Epoch 10/200: 100%|█| 459/459 [01:23<00:00,  5.51it/s, D=0.420, G_adv=3.224, CLIP=0.778, G_tot=3.380
Epoch 11/200: 100%|█| 459/459 [01:23<00:00,  5.52it/s, D=0.35

## Scoring Checkpoints

In [6]:
!python select_best_gen.py \
  --cfg cfg/flowers_clip.yml \
  --ckpt_dir "$DRIVE_PROJ/output_clip/checkpoints" \
  --prompt "a bright yellow sunflower with a dark center" \
  --n 64 \
  --trunc 0.7


Found 198 generator checkpoints.
Using device: cuda
Scoring checkpoints...
Epoch   1: CLIP score = 0.2110
Epoch   2: CLIP score = 0.2288
Epoch   3: CLIP score = 0.2256
Epoch   4: CLIP score = 0.2294
Epoch   5: CLIP score = 0.2220
Epoch   6: CLIP score = 0.2246
Epoch   7: CLIP score = 0.2211
Epoch  10: CLIP score = 0.2187
Epoch  11: CLIP score = 0.2220
Epoch  12: CLIP score = 0.2263
Epoch  13: CLIP score = 0.2255
Epoch  14: CLIP score = 0.2237
Epoch  15: CLIP score = 0.2278
Epoch  16: CLIP score = 0.2274
Epoch  17: CLIP score = 0.2250
Epoch  18: CLIP score = 0.2262
Epoch  19: CLIP score = 0.2225
Epoch  20: CLIP score = 0.2278
Epoch  21: CLIP score = 0.2204
Epoch  22: CLIP score = 0.2248
Epoch  23: CLIP score = 0.2251
Epoch  24: CLIP score = 0.2283
Epoch  25: CLIP score = 0.2271
Epoch  26: CLIP score = 0.2320
Epoch  27: CLIP score = 0.2297
Epoch  28: CLIP score = 0.2283
Epoch  29: CLIP score = 0.2204
Epoch  30: CLIP score = 0.2200
Epoch  31: CLIP score = 0.2225
Epoch  32: CLIP score = 0.

## Inference

In [8]:
!python tools/make_real_split.py \
  --jpg_root ./102flowers/jpg \
  --val_classes ./102flowers/valclasses.txt \
  --out_dir output_clip/eval/val_64 \
  --image_size 64


Resizing val images: 100% 8189/8189 [03:56<00:00, 34.65it/s]


In [14]:
%cd /content/drive/MyDrive/DCGAN-Text-to-Image
!touch tools/__init__.py
!python -u tools/eval_clip_over_epoch.py \
  --ckpt_dir output_clip/checkpoints \
  --cfg cfg/flowers_clip.yml \
  --prompts_file output_clip/eval/prompts.yaml \
  --samples_per_prompt 4 \
  --trunc 0.9 \
  --summary_csv output_clip/logs/eval_clip_summary.csv \
  --samples_csv output_clip/logs/eval_clip_samples_all.csv


/content/drive/MyDrive/DCGAN-Text-to-Image
Starting eval_clip_over_epoch_safe...
ckpt_dir = output_clip/checkpoints
Device: cuda
[1/198] Loading gen_001.pth
  mean CLIP score = 0.2127  (0.6s)
[2/198] Loading gen_002.pth
  mean CLIP score = 0.2314  (0.3s)
[3/198] Loading gen_003.pth
  mean CLIP score = 0.2283  (0.3s)
[4/198] Loading gen_004.pth
  mean CLIP score = 0.2307  (0.3s)
[5/198] Loading gen_005.pth
  mean CLIP score = 0.2319  (0.3s)
[6/198] Loading gen_006.pth
  mean CLIP score = 0.2245  (0.3s)
[7/198] Loading gen_007.pth
  mean CLIP score = 0.2276  (0.3s)
[8/198] Loading gen_010.pth
  mean CLIP score = 0.2317  (3.3s)
[9/198] Loading gen_011.pth
  mean CLIP score = 0.2292  (0.3s)
[10/198] Loading gen_012.pth
  mean CLIP score = 0.2357  (0.3s)
[11/198] Loading gen_013.pth
  mean CLIP score = 0.2341  (0.3s)
[12/198] Loading gen_014.pth
  mean CLIP score = 0.2333  (0.4s)
[13/198] Loading gen_015.pth
  mean CLIP score = 0.2343  (0.3s)
[14/198] Loading gen_016.pth
  mean CLIP score =

In [19]:

%cd /content/drive/MyDrive/DCGAN-Text-to-Image

E = 195
E3 = f"{E:03d}"


!mkdir -p tools; touch tools/__init__.py


# Sanity checks
!ls -l output_clip/checkpoints/gen_{E3}.pth
!ls -l output_clip/checkpoints/genEMA_{E3}.pth 2>/dev/null || echo "No EMA checkpoint for epoch {E3}"
!ls -l output_clip/eval/prompts.yaml || echo "Missing prompts.yaml"
!ls -ld output_clip/eval/val_64 2>/dev/null || echo "Missing real_dir (val_64)."


/content/drive/MyDrive/DCGAN-Text-to-Image
-rw------- 1 root root 18803803 Nov 13 06:30 output_clip/checkpoints/gen_195.pth
-rw------- 1 root root 18802997 Nov 13 06:30 output_clip/checkpoints/genEMA_195.pth
-rw------- 1 root root 581 Nov 13 05:49 output_clip/eval/prompts.yaml
drwx------ 2 root root 4096 Nov 13 12:14 output_clip/eval/val_64


In [20]:
# Build the command in Python so there’s zero ambiguity
base = (
    f"python -m tools.eval_fid_for_epoch "
    f"--gen_ckpt output_clip/checkpoints/gen_{E3}.pth "
    f"--cfg cfg/flowers_clip.yml "
    f"--prompts_file output_clip/eval/prompts.yaml "
    f"--n_imgs 1020 "
    f"--trunc 0.9 "
    f"--real_dir output_clip/eval/val_64 "
    f"--fake_dir output_clip/eval/fake_epoch{E3} "
    f"--out_txt output_clip/logs/fid/fid_epoch{E3}.txt "
)

# Append EMA if it exists
cmd = base
if os.path.exists(f"output_clip/checkpoints/genEMA_{E3}.pth"):
    cmd += f"--ema_ckpt output_clip/checkpoints/genEMA_{E3}.pth"

print(cmd)   # for visibility
!{cmd}


python -m tools.eval_fid_for_epoch --gen_ckpt output_clip/checkpoints/gen_195.pth --cfg cfg/flowers_clip.yml --prompts_file output_clip/eval/prompts.yaml --n_imgs 1020 --trunc 0.9 --real_dir output_clip/eval/val_64 --fake_dir output_clip/eval/fake_epoch195 --out_txt output_clip/logs/fid/fid_epoch195.txt --ema_ckpt output_clip/checkpoints/genEMA_195.pth
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:01<00:00, 58.0MB/s]
/usr/local/lib/python3.12/dist-packages/torch_fidelity/datasets.py:16: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.fr

In [21]:
E = 175
E3 = f"{E:03d}"
# Build the command in Python so there’s zero ambiguity
base = (
    f"python -m tools.eval_fid_for_epoch "
    f"--gen_ckpt output_clip/checkpoints/gen_{E3}.pth "
    f"--cfg cfg/flowers_clip.yml "
    f"--prompts_file output_clip/eval/prompts.yaml "
    f"--n_imgs 1020 "
    f"--trunc 0.9 "
    f"--real_dir output_clip/eval/val_64 "
    f"--fake_dir output_clip/eval/fake_epoch{E3} "
    f"--out_txt output_clip/logs/fid/fid_epoch{E3}.txt "
)

# Append EMA if it exists
cmd = base
if os.path.exists(f"output_clip/checkpoints/genEMA_{E3}.pth"):
    cmd += f"--ema_ckpt output_clip/checkpoints/genEMA_{E3}.pth"

print(cmd)   # for visibility
!{cmd}


python -m tools.eval_fid_for_epoch --gen_ckpt output_clip/checkpoints/gen_175.pth --cfg cfg/flowers_clip.yml --prompts_file output_clip/eval/prompts.yaml --n_imgs 1020 --trunc 0.9 --real_dir output_clip/eval/val_64 --fake_dir output_clip/eval/fake_epoch175 --out_txt output_clip/logs/fid/fid_epoch175.txt --ema_ckpt output_clip/checkpoints/genEMA_175.pth
/usr/local/lib/python3.12/dist-packages/torch_fidelity/datasets.py:16: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes())).view(height, width, 3)


In [27]:
!python -m tools.eval_lpips_diversity \
  --ckpt_dir output_clip/checkpoints \
  --cfg cfg/flowers_clip.yml \
  --prompts_file output_clip/eval/prompts.yaml \
  --samples_per_prompt 8 \
  --trunc 0.9 \
  --out_csv output_clip/logs/eval_lpips_diversity.csv \
  --ema


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100% 233M/233M [00:01<00:00, 230MB/s]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
[INFO] Found 198 checkpoints in output_cli

In [29]:
E = 175
gen_ckpt  = f"output_clip/checkpoints/gen_{E:03d}.pth"
ema_ckpt  = f"output_clip/checkpoints/genEMA_{E:03d}.pth"
out_csv   = f"output_clip/logs/class/zero_shot_epoch{E:03d}.csv"

for p in [gen_ckpt, ema_ckpt]:
    print(p, "=>", os.path.exists(p))


output_clip/checkpoints/gen_175.pth => True
output_clip/checkpoints/genEMA_175.pth => True


In [30]:
import sys, subprocess
sys.path.insert(0, "/content/drive/MyDrive/DCGAN-Text-to-Image")  # so `tools` package imports
cmd = [
  "python", "-m", "tools.eval_zero_shot_class",
  "--gen_ckpt", gen_ckpt,
  "--cfg", "cfg/flowers_clip.yml",
  "--class_map_file", "class_names.py",
  "--n_per_class", "8",
  "--trunc", "0.9",
  "--out_csv", out_csv,
  "--ema_ckpt", ema_ckpt,  # remove these two args if EMA file is missing
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


python -m tools.eval_zero_shot_class --gen_ckpt output_clip/checkpoints/gen_175.pth --cfg cfg/flowers_clip.yml --class_map_file class_names.py --n_per_class 8 --trunc 0.9 --out_csv output_clip/logs/class/zero_shot_epoch175.csv --ema_ckpt output_clip/checkpoints/genEMA_175.pth


CompletedProcess(args=['python', '-m', 'tools.eval_zero_shot_class', '--gen_ckpt', 'output_clip/checkpoints/gen_175.pth', '--cfg', 'cfg/flowers_clip.yml', '--class_map_file', 'class_names.py', '--n_per_class', '8', '--trunc', '0.9', '--out_csv', 'output_clip/logs/class/zero_shot_epoch175.csv', '--ema_ckpt', 'output_clip/checkpoints/genEMA_175.pth'], returncode=0)

In [31]:
%%bash
cat > tools/export_epoch_prompts.py <<'PY'
import os, sys, argparse, yaml, math, random
from datetime import datetime
import torch, torch.nn.functional as F
from torchvision.utils import make_grid, save_image
from PIL import Image
import clip
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from models.clip_gan import Generator

def set_seed(s):
    random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def read_prompts_yaml(p):
    with open(p, "r") as f:
        y = yaml.safe_load(f)
    return [str(x) for x in y.get("prompts", [])]

def slugify(t):
    s = "".join(c.lower() if c.isalnum() else "_" for c in t).strip("_")
    return "_".join(s.split("_")[:10])[:80] or "prompt"

@torch.no_grad()
def encode_texts(model, device, texts):
    toks = clip.tokenize(texts, truncate=True).to(device)
    feats = model.encode_text(toks).float()
    return feats / feats.norm(dim=-1, keepdim=True)

@torch.no_grad()
def clip_scores(model, device, imgs, text_feats):
    imgs_01 = (imgs + 1) / 2
    imgs_224 = F.interpolate(imgs_01, size=224, mode="bilinear", align_corners=False)
    imgf = model.encode_image(imgs_224).float()
    imgf = imgf / imgf.norm(dim=-1, keepdim=True)
    return (imgf * text_feats).sum(dim=-1)

def load_generator(gen_ckpt, image_size, z_dim, gf_dim, device, ema_ckpt=None):
    G = Generator(z_dim=z_dim, cond_dim=512, image_size=image_size, gf_dim=gf_dim).to(device)
    sd = torch.load(gen_ckpt, map_location=device)
    G.load_state_dict(sd, strict=True)
    if ema_ckpt and os.path.exists(ema_ckpt):
        ema_sd = torch.load(ema_ckpt, map_location=device)
        G.load_state_dict(ema_sd, strict=False)
    G.eval()
    return G

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", required=True)
    ap.add_argument("--ckpt_dir", required=True)
    ap.add_argument("--epochs", required=True)  # e.g. "175,120,176,122"
    ap.add_argument("--prompts_file", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--n_per_prompt", type=int, default=64)
    ap.add_argument("--topk", type=int, default=1)
    ap.add_argument("--trunc", type=float, default=0.7)
    ap.add_argument("--seed", type=int, default=1337)
    ap.add_argument("--ema", action="store_true")
    args = ap.parse_args()

    # Minimal cfg loader
    import yaml as _y
    with open(args.cfg, "r") as f:
        cfg = _y.safe_load(f)
    image_size = int(cfg.get("IMAGE_SIZE", 64))
    z_dim      = int(cfg.get("Z_DIM", 100))
    gf_dim     = int(cfg.get("GF_DIM", 64))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(args.seed)

    os.makedirs(args.out_dir, exist_ok=True)
    prompts = read_prompts_yaml(args.prompts_file)
    assert len(prompts) > 0

    clip_model, _ = clip.load("ViT-B/32", device=device)
    clip_model.eval()
    clip_model = clip_model.float()

    epochs = [int(e.strip()) for e in args.epochs.split(",") if e.strip()]
    for e in epochs:
        gen_p = os.path.join(args.ckpt_dir, f"gen_{e:03d}.pth")
        ema_p = os.path.join(args.ckpt_dir, f"genEMA_{e:03d}.pth") if args.ema else None
        if not os.path.exists(gen_p):
            print(f"[skip] {gen_p} not found")
            continue

        G = load_generator(gen_p, image_size, z_dim, gf_dim, device, ema_ckpt=ema_p if (ema_p and os.path.exists(ema_p)) else None)

        epoch_dir = os.path.join(args.out_dir, f"epoch{e:03d}")
        os.makedirs(epoch_dir, exist_ok=True)

        best_images = []
        for ptxt in prompts:
            tf = encode_texts(clip_model, device, [ptxt])  # [1,512]
            z = torch.randn(args.n_per_prompt, z_dim, device=device) * args.trunc
            tf_rep = tf.repeat(args.n_per_prompt, 1)

            imgs = G(z, tf_rep).clamp(-1, 1)  # [N,3,H,W]
            sc = clip_scores(clip_model, device, imgs, tf_rep)  # [N]

            topk = min(args.topk, imgs.size(0))
            vals, idx = torch.topk(sc, k=topk, largest=True)
            chosen = imgs[idx]  # [topk,3,H,W]

            # Save per-prompt best image (first of top-k)
            prompt_png = os.path.join(epoch_dir, f"{slugify(ptxt)}.png")
            save_image((chosen[0] + 1)/2, prompt_png)
            best_images.append(chosen[0])

        # Combined grid for the epoch
        grid = make_grid(torch.stack(best_images, dim=0), nrow=math.ceil(len(prompts)/2))
        grid_png = os.path.join(args.out_dir, f"epoch{e:03d}_grid.png")
        save_image((grid + 1)/2, grid_png)
        print(f"[done] epoch {e:03d}: saved {len(prompts)} prompt images and {grid_png}")

if __name__ == "__main__":
    main()
PY


In [32]:
%cd /content/drive/MyDrive/DCGAN-Text-to-Image
!PYTHONPATH=. python -m tools.export_epoch_prompts \
  --cfg cfg/flowers_clip.yml \
  --ckpt_dir output_clip/checkpoints \
  --epochs 175,120,176,122,172,190,109,96,99,93 \
  --prompts_file output_clip/eval/prompts.yaml \
  --out_dir output_clip/vis \
  --n_per_prompt 64 \
  --topk 1 \
  --trunc 0.7 \
  --ema


/content/drive/MyDrive/DCGAN-Text-to-Image
[done] epoch 175: saved 12 prompt images and output_clip/vis/epoch175_grid.png
[done] epoch 120: saved 12 prompt images and output_clip/vis/epoch120_grid.png
[done] epoch 176: saved 12 prompt images and output_clip/vis/epoch176_grid.png
[done] epoch 122: saved 12 prompt images and output_clip/vis/epoch122_grid.png
[done] epoch 172: saved 12 prompt images and output_clip/vis/epoch172_grid.png
[done] epoch 190: saved 12 prompt images and output_clip/vis/epoch190_grid.png
[done] epoch 109: saved 12 prompt images and output_clip/vis/epoch109_grid.png
[done] epoch 096: saved 12 prompt images and output_clip/vis/epoch096_grid.png
[done] epoch 099: saved 12 prompt images and output_clip/vis/epoch099_grid.png
[done] epoch 093: saved 12 prompt images and output_clip/vis/epoch093_grid.png


In [33]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ====== Paths ======
BASE = "/content/drive/MyDrive/DCGAN-Text-to-Image"
LOG_DIR = os.path.join(BASE, "output_clip", "logs")
EVAL_DIR = os.path.join(BASE, "output_clip", "eval")
FIG_DIR = os.path.join(BASE, "output_clip", "figs")
os.makedirs(FIG_DIR, exist_ok=True)


TRAIN_CSV_PATTERN = os.path.join(LOG_DIR, "train_*.csv")             # train files saved with default naming train_20251113_020008.csv
CLIP_SUMMARY_CSV = os.path.join(LOG_DIR, "eval_clip_summary.csv")
CLIP_SAMPLES_CSV = os.path.join(LOG_DIR, "eval_clip_samples_all.csv")
LPIPS_DIV_CSV    = os.path.join(LOG_DIR, "eval_lpips_diversity.csv")

# ====== Helpers ======
def find_latest_csv(pattern):
    files = glob.glob(pattern)
    if not files:
        return None
    files = sorted(files, key=lambda p: os.path.getmtime(p))
    return files[-1]

def smooth_series(s, window=5):
    return s.rolling(window=window, min_periods=1, center=True).mean()

def safe_read_csv(path):
    if path and os.path.exists(path):
        try:
            return pd.read_csv(path)
        except Exception:
            pass
    return None

def get_first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of the columns found: {candidates}")

# ====== 1) Training losses over epochs ======
train_csv = find_latest_csv(TRAIN_CSV_PATTERN)
if train_csv:
    df_tr = safe_read_csv(train_csv)
    if df_tr is not None and len(df_tr) > 0:
        try:
            epoch_col = get_first_col(df_tr, ["epoch", "Epoch"])
            d_col     = get_first_col(df_tr, ["d_loss", "D", "disc_loss"])
            gadv_col  = get_first_col(df_tr, ["g_adv", "G_adv", "gen_adv"])
            clip_col  = get_first_col(df_tr, ["clip_loss", "CLIP", "clip"])
            gtot_col  = get_first_col(df_tr, ["g_total", "G_tot", "gen_total"])

            x = df_tr[epoch_col].astype(int)
            plt.figure(figsize=(8, 5))
            plt.plot(x, smooth_series(df_tr[d_col]), label="D loss")
            plt.plot(x, smooth_series(df_tr[gadv_col]), label="G adv")
            plt.plot(x, smooth_series(df_tr[clip_col]), label="CLIP loss")
            plt.plot(x, smooth_series(df_tr[gtot_col]), label="G total")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.title("Training losses over epochs")
            plt.legend()
            plt.tight_layout()
            out = os.path.join(FIG_DIR, "fig_training_losses.png")
            plt.savefig(out, dpi=150)
            plt.close()
        except Exception as e:
            print("Training plot skipped:", e)
else:
    print("No training CSV found; skipped training-loss plot.")

# ====== 2) CLIP alignment vs. epoch ======
df_clip_sum = safe_read_csv(CLIP_SUMMARY_CSV)
if df_clip_sum is None or len(df_clip_sum) == 0:
    df_clip_samples = safe_read_csv(CLIP_SAMPLES_CSV)
    if df_clip_samples is not None and len(df_clip_samples) > 0:
        try:
            epc = get_first_col(df_clip_samples, ["epoch", "Epoch"])
            sc  = get_first_col(df_clip_samples, ["clip_score", "clip", "score"])
            df_clip_sum = df_clip_samples.groupby(epc)[sc].agg(clip_mean="mean", clip_std="std", n="count").reset_index()
            df_clip_sum.rename(columns={epc: "epoch"}, inplace=True)
        except Exception as e:
            df_clip_sum = None
            print("Could not build CLIP summary from samples:", e)

if df_clip_sum is not None and len(df_clip_sum) > 0:
    try:
        epc = get_first_col(df_clip_sum, ["epoch", "Epoch"])
        mean_col = get_first_col(df_clip_sum, ["clip_mean", "mean_clip", "clip_score_mean"])
        plt.figure(figsize=(8, 5))
        plt.plot(df_clip_sum[epc].astype(int), smooth_series(df_clip_sum[mean_col]), label="CLIP alignment (mean)")
        plt.xlabel("Epoch")
        plt.ylabel("CLIP similarity")
        plt.title("CLIP alignment over epochs")
        plt.legend()
        plt.tight_layout()
        out = os.path.join(FIG_DIR, "fig_clip_alignment_over_epochs.png")
        plt.savefig(out, dpi=150)
        plt.close()
    except Exception as e:
        print("CLIP-over-epochs plot skipped:", e)
else:
    print("No CLIP summary available; skipped CLIP-over-epochs plot.")

# ====== 3) Per-prompt CLIP alignment (boxplot) ======
df_clip_samples = safe_read_csv(CLIP_SAMPLES_CSV)
if df_clip_samples is not None and len(df_clip_samples) > 0:
    try:
        prompt_col = get_first_col(df_clip_samples, ["prompt", "text", "caption"])
        score_col  = get_first_col(df_clip_samples, ["clip_score", "clip", "score"])
        counts = df_clip_samples[prompt_col].value_counts()
        top_prompts = counts.head(12).index.tolist()
        data = [df_clip_samples.loc[df_clip_samples[prompt_col] == p, score_col].dropna().values for p in top_prompts]
        labels = [str(p)[:40] + ("…" if len(str(p)) > 40 else "") for p in top_prompts]

        plt.figure(figsize=(10, 6))
        plt.boxplot(data, labels=labels, showfliers=False)
        plt.xticks(rotation=30, ha="right")
        plt.ylabel("CLIP similarity")
        plt.title("Per-prompt CLIP alignment (top prompts)")
        plt.tight_layout()
        out = os.path.join(FIG_DIR, "fig_clip_per_prompt_boxplot.png")
        plt.savefig(out, dpi=150)
        plt.close()
    except Exception as e:
        print("Per-prompt boxplot skipped:", e)
else:
    print("No per-sample CLIP CSV found; skipped per-prompt boxplot.")

# ====== 4) CLIP alignment histogram ======
if df_clip_samples is not None and len(df_clip_samples) > 0:
    try:
        score_col  = get_first_col(df_clip_samples, ["clip_score", "clip", "score"])
        plt.figure(figsize=(8, 5))
        plt.hist(df_clip_samples[score_col].dropna().values, bins=30)
        plt.xlabel("CLIP similarity")
        plt.ylabel("Count")
        plt.title("Distribution of per-sample CLIP similarity")
        plt.tight_layout()
        out = os.path.join(FIG_DIR, "fig_clip_histogram.png")
        plt.savefig(out, dpi=150)
        plt.close()
    except Exception as e:
        print("CLIP histogram skipped:", e)

# ====== 5) LPIPS diversity vs. epoch ======
df_lp = safe_read_csv(LPIPS_DIV_CSV)
if df_lp is not None and len(df_lp) > 0:
    try:
        epc = get_first_col(df_lp, ["epoch", "Epoch"])
        lp_mean = get_first_col(df_lp, ["lpips_mean", "lpips", "mean_lpips"])
        plt.figure(figsize=(8, 5))
        plt.plot(df_lp[epc].astype(int), smooth_series(df_lp[lp_mean]), label="LPIPS (mean)")
        plt.xlabel("Epoch")
        plt.ylabel("LPIPS")
        plt.title("LPIPS diversity over epochs")
        plt.legend()
        plt.tight_layout()
        out = os.path.join(FIG_DIR, "fig_lpips_over_epochs.png")
        plt.savefig(out, dpi=150)
        plt.close()
    except Exception as e:
        print("LPIPS plot skipped:", e)
else:
    print("No LPIPS CSV found; skipped LPIPS plot.")

print("Done. Figures saved to:", FIG_DIR)


CLIP-over-epochs plot skipped: "None of the columns found: ['clip_mean', 'mean_clip', 'clip_score_mean']"


/tmp/ipython-input-3998624873.py:120: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


LPIPS plot skipped: "None of the columns found: ['lpips_mean', 'lpips', 'mean_lpips']"
Done. Figures saved to: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs


In [38]:
# ===  plotting for training, CLIP, and LPIPS CSVs ===


import os, re, glob, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "/content/drive/MyDrive/DCGAN-Text-to-Image"
LOG_DIR = os.path.join(BASE, "output_clip", "logs")
FIG_DIR = os.path.join(BASE, "output_clip", "figs")
os.makedirs(FIG_DIR, exist_ok=True)

# Default file locations
TRAIN_CSV          = None  # will auto-pick latest train_*.csv
CLIP_SUMMARY_CSV   = os.path.join(LOG_DIR, "eval_clip_summary.csv")
CLIP_SAMPLES_CSV   = os.path.join(LOG_DIR, "eval_clip_samples_all.csv")
LPIPS_CSV          = os.path.join(LOG_DIR, "eval_lpips_diversity.csv")

# Auto pick latest train csv if present
train_candidates = sorted(glob.glob(os.path.join(LOG_DIR, "train_*.csv")))
if train_candidates:
    TRAIN_CSV = train_candidates[-1]

def pick_col(df, candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the columns found: {candidates}")
    return None

def parse_epoch_inplace(df):
    # If "epoch" already exists, keep it
    if "epoch" in df.columns:
        return df
    # Try to derive from a path-like column
    for c in ["ckpt","checkpoint","gen_ckpt","file","path","checkpoint_path","model","src"]:
        if c in df.columns:
            def parse_one(s):
                m = re.search(r'gen[_\-]?(\d{1,5})\.pth', str(s))
                return int(m.group(1)) if m else None
            df = df.copy()
            df["epoch"] = df[c].map(parse_one)
            return df
    return df

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    print("Saved:", path)

# ---------- 1) Training curves ----------
def plot_training_curves(train_csv):
    if not train_csv or not os.path.isfile(train_csv):
        print("Training curves: CSV not found -> skipped.")
        return
    df = pd.read_csv(train_csv)
    if "epoch" not in df.columns:
        print("Training curves: 'epoch' column missing -> skipped.")
        return
    # Trying sensible defaults and fallbacks
    d_col   = pick_col(df, ["d_loss","D","disc_loss"])
    gadv    = pick_col(df, ["g_adv","G_adv","gen_adv","g_loss"])
    clipc   = pick_col(df, ["clip_loss","CLIP","clip"])
    gtot    = pick_col(df, ["g_total","G_tot","gen_total"])

    # Plot any that exist
    ycols = []
    labels = []
    if d_col:   ycols.append(d_col);   labels.append("Discriminator Loss")
    if gadv:    ycols.append(gadv);    labels.append("Generator Adv Loss")
    if clipc:   ycols.append(clipc);   labels.append("CLIP Alignment Loss")
    if gtot:    ycols.append(gtot);    labels.append("Generator Total Loss")
    if not ycols:
        print("Training curves: no known loss columns -> skipped.")
        return
    plt.figure(figsize=(8,5))
    for c, lab in zip(ycols, labels):
        plt.plot(df["epoch"], df[c], label=lab)
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training losses over epochs")
    plt.legend()
    savefig(os.path.join(FIG_DIR, "fig_training_losses.png"))

# ---------- 2) CLIP plots (over epochs, histogram, per-prompt box) ----------
def plot_clip_metrics(summary_csv, samples_csv):
    df_sum = None
    if summary_csv and os.path.isfile(summary_csv):
        try:
            df_sum = pd.read_csv(summary_csv)
        except Exception as e:
            print("CLIP summary read failed:", e)

    # summary for epoch curve
    used = None
    df_curve = None
    if df_sum is not None and len(df_sum):
        df_sum = parse_epoch_inplace(df_sum)
        mean_col = pick_col(df_sum, ["clip_mean","mean_clip","clip_score_mean","mean","avg","avg_clip"])
        if "epoch" in df_sum.columns and mean_col:
            used = "summary"
            df_curve = df_sum[["epoch", mean_col]].rename(columns={mean_col:"clip_mean"}).dropna()

    # , build epoch curve from samples
    if df_curve is None:
        if not (samples_csv and os.path.isfile(samples_csv)):
            print("CLIP over-epochs: no usable summary and no samples CSV -> skipped curve.")
        else:
            try:
                dfs = pd.read_csv(samples_csv)
                if len(dfs):
                    dfs = parse_epoch_inplace(dfs)
                    score_col = pick_col(dfs, ["clip_score","clip","similarity","sim"], required=True)
                    if "epoch" in dfs.columns:
                        df_curve = (dfs.dropna(subset=["epoch"])
                                      .groupby("epoch")[score_col]
                                      .agg(clip_mean="mean", clip_std="std", n="count")
                                      .reset_index())
                        used = "samples"
            except Exception as e:
                print("CLIP curve from samples failed:", e)

    # Plot the curve if ready
    if df_curve is not None and len(df_curve):
        df_curve = df_curve.sort_values("epoch")
        plt.figure(figsize=(8,5))
        plt.plot(df_curve["epoch"], df_curve["clip_mean"], marker="o")
        if "clip_std" in df_curve.columns and not df_curve["clip_std"].isna().all():
            e = df_curve["epoch"].values
            m = df_curve["clip_mean"].values
            s = df_curve["clip_std"].values
            plt.fill_between(e, m - s, m + s, alpha=0.2)
        plt.xlabel("Epoch"); plt.ylabel("CLIP similarity")
        plt.title(f"CLIP alignment over epochs ({used})")
        savefig(os.path.join(FIG_DIR, "fig_clip_over_epochs.png"))
    else:
        print("CLIP over-epochs plot skipped: could not form an epoch->mean mapping.")

    # histogram + per-prompt box from samples only
    if not (samples_csv and os.path.isfile(samples_csv)):
        print("CLIP histogram/box: samples CSV not found -> skipped.")
        return
    try:
        dfs = pd.read_csv(samples_csv)
    except Exception as e:
        print("CLIP samples read failed:", e); return
    if len(dfs) == 0:
        print("CLIP samples CSV empty -> skipped histogram/box.")
        return

    score_col = pick_col(dfs, ["clip_score","clip","similarity","sim"])
    if not score_col:
        print("CLIP samples: no clip score column -> skipped histogram/box.")
        return

    # Histogram
    plt.figure(figsize=(7,5))
    vals = dfs[score_col].dropna().values
    plt.hist(vals, bins=30)
    plt.xlabel("CLIP similarity"); plt.ylabel("Count")
    plt.title("Distribution of CLIP similarity (all generated samples)")
    savefig(os.path.join(FIG_DIR, "fig_clip_histogram.png"))

    # Per-prompt box
    prompt_col = pick_col(dfs, ["prompt","text","caption"])
    if prompt_col:
        groups = dfs.dropna(subset=[score_col]).groupby(prompt_col)[score_col]
        labels, data = [], []
        for k, g in groups:
            labels.append(textwrap.shorten(str(k), width=48))
            data.append(g.values)
        if len(data) >= 2:
            plt.figure(figsize=(10, max(4, 0.28*len(labels))))
            plt.boxplot(data, showfliers=False)
            plt.xticks(range(1, len(labels)+1), labels, rotation=60, ha="right")
            plt.ylabel("CLIP similarity")
            plt.title("CLIP similarity per prompt")
            savefig(os.path.join(FIG_DIR, "fig_clip_per_prompt_boxplot.png"))
        else:
            print("CLIP boxplot: need >=2 prompts to make a box plot -> skipped.")
    else:
        print("CLIP boxplot: no prompt/text column -> skipped.")

# ---------- 3) LPIPS diversity (over epochs + histogram) ----------
def plot_lpips(lpips_csv):
    if not lpips_csv or not os.path.isfile(lpips_csv):
        print("LPIPS: CSV not found -> skipped.")
        return
    try:
        df = pd.read_csv(lpips_csv)
    except Exception as e:
        print("LPIPS read failed:", e); return
    if len(df) == 0:
        print("LPIPS CSV empty -> skipped.")
        return

    df = parse_epoch_inplace(df)
    lp_col = pick_col(df, ["lpips","lpips_mean","mean_lpips","distance"])
    if lp_col is None:
        print("LPIPS: no recognizable LPIPS column -> skipped.")
        return


    if "epoch" in df.columns:
        if lp_col.endswith("_mean") or "mean" in lp_col:
            dplot = df[["epoch", lp_col]].rename(columns={lp_col:"lpips_mean"}).dropna()
        else:
            dplot = (df.dropna(subset=["epoch"])
                       .groupby("epoch")[lp_col]
                       .agg(lpips_mean="mean", lpips_std="std", n="count")
                       .reset_index())
        dplot = dplot.sort_values("epoch")
        plt.figure(figsize=(8,5))
        plt.plot(dplot["epoch"], dplot["lpips_mean"], marker="o")
        if "lpips_std" in dplot.columns and not dplot["lpips_std"].isna().all():
            e = dplot["epoch"].values
            m = dplot["lpips_mean"].values
            s = dplot["lpips_std"].values
            plt.fill_between(e, m - s, m + s, alpha=0.2)
        plt.xlabel("Epoch"); plt.ylabel("LPIPS (lower is more similar)")
        plt.title("LPIPS diversity over epochs")
        savefig(os.path.join(FIG_DIR, "fig_lpips_over_epochs.png"))
    else:
        print("LPIPS over-epochs: cannot find/infer epoch -> skipping line plot.")

    # Histogram (all pairs)
    plt.figure(figsize=(7,5))
    plt.hist(df[lp_col].dropna().values, bins=30)
    plt.xlabel("LPIPS"); plt.ylabel("Count")
    plt.title("Distribution of LPIPS distances (all prompts/pairs)")
    savefig(os.path.join(FIG_DIR, "fig_lpips_histogram.png"))

# ---------- Run all ----------
print("Found:")
print("  TRAIN_CSV       :", TRAIN_CSV)
print("  CLIP_SUMMARY_CSV:", CLIP_SUMMARY_CSV)
print("  CLIP_SAMPLES_CSV:", CLIP_SAMPLES_CSV)
print("  LPIPS_CSV       :", LPIPS_CSV)

plot_training_curves(TRAIN_CSV)
plot_clip_metrics(CLIP_SUMMARY_CSV, CLIP_SAMPLES_CSV)
plot_lpips(LPIPS_CSV)

print("All done. Check:", FIG_DIR)


Found:
  TRAIN_CSV       : /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/logs/train_20251113_020008.csv
  CLIP_SUMMARY_CSV: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/logs/eval_clip_summary.csv
  CLIP_SAMPLES_CSV: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/logs/eval_clip_samples_all.csv
  LPIPS_CSV       : /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/logs/eval_lpips_diversity.csv
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_training_losses.png
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_clip_over_epochs.png
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_clip_histogram.png
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_clip_per_prompt_boxplot.png
LPIPS: no recognizable LPIPS column -> skipped.
All done. Check: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs


In [40]:
# === LPIPS plotting  ===
import os, glob, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = "/content/drive/MyDrive/DCGAN-Text-to-Image"
LOG_DIR = os.path.join(BASE, "output_clip", "logs")
FIG_DIR = os.path.join(BASE, "output_clip", "figs")
os.makedirs(FIG_DIR, exist_ok=True)

LPIPS_CSV  = os.path.join(LOG_DIR, "eval_lpips_diversity.csv")
LPIPS_XLSX = os.path.join(LOG_DIR, "eval_lpips_diversity.xlsx")

def pick_col(df, candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the columns found: {candidates}")
    return None

def parse_epoch_inplace(df):
    if "epoch" in df.columns:
        return df
    for c in ["ckpt","checkpoint","gen_ckpt","file","path","checkpoint_path","model","src"]:
        if c in df.columns:
            def parse_one(s):
                m = re.search(r'gen[_\-]?(\d{1,5})\.pth', str(s))
                return int(m.group(1)) if m else None
            df = df.copy()
            df["epoch"] = df[c].map(parse_one)
            return df
    for c in ["Epoch","EPOCH","ep"]:
        if c in df.columns:
            df = df.copy()
            df["epoch"] = pd.to_numeric(df[c], errors="coerce")
            return df
    return df

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    print("Saved:", path)

# --- Detect and load LPIPS table ---
def load_lpips_table():
    # Prefer CSV
    if os.path.isfile(LPIPS_CSV):
        print("Using LPIPS CSV:", LPIPS_CSV)
        return pd.read_csv(LPIPS_CSV)
    if os.path.isfile(LPIPS_XLSX):
        print("Converting XLSX -> CSV:", LPIPS_XLSX)
        df = pd.read_excel(LPIPS_XLSX, engine="openpyxl")
        df.to_csv(LPIPS_CSV, index=False)
        return df
    xlsx_candidates = sorted(glob.glob(os.path.join(LOG_DIR, "**", "*lpips*.xlsx"), recursive=True))
    if xlsx_candidates:
        xfile = xlsx_candidates[-1]
        print("Converting XLSX -> CSV:", xfile)
        df = pd.read_excel(xfile, engine="openpyxl")
        df.to_csv(LPIPS_CSV, index=False)
        return df

    csv_candidates = sorted(glob.glob(os.path.join(LOG_DIR, "**", "*lpips*.csv"), recursive=True))
    if csv_candidates:
        cfile = csv_candidates[-1]
        print("Using LPIPS CSV:", cfile)
        return pd.read_csv(cfile)
    print("LPIPS table not found.")
    return None

def plot_lpips(df):
    if df is None or len(df) == 0:
        print("LPIPS: empty or missing -> skipped.")
        return

    print("LPIPS columns:", list(df.columns))
    df = parse_epoch_inplace(df)

    lp_col = pick_col(df, ["lpips","lpips_mean","mean_lpips","distance","lpips_value"])
    if lp_col is None:
        print("LPIPS: no recognizable column -> skipped.")
        return

    if "epoch" in df.columns and not df["epoch"].isna().all():
        if lp_col.endswith("_mean") or "mean" in lp_col:
            dplot = df[["epoch", lp_col]].rename(columns={lp_col:"lpips_mean"}).dropna()
        else:
            dplot = (df.dropna(subset=["epoch"])
                       .groupby("epoch")[lp_col]
                       .agg(lpips_mean="mean", lpips_std="std", n="count")
                       .reset_index())
        if len(dplot):
            dplot = dplot.sort_values("epoch")
            plt.figure(figsize=(8,5))
            plt.plot(dplot["epoch"], dplot["lpips_mean"], marker="o")
            if "lpips_std" in dplot.columns and not dplot["lpips_std"].isna().all():
                e = dplot["epoch"].values
                m = dplot["lpips_mean"].values
                s = dplot["lpips_std"].values
                plt.fill_between(e, m - s, m + s, alpha=0.2)
            plt.xlabel("Epoch"); plt.ylabel("LPIPS (lower means images are more similar)")
            plt.title("LPIPS diversity over epochs")
            savefig(os.path.join(FIG_DIR, "fig_lpips_over_epochs.png"))
        else:
            print("LPIPS over-epochs: no rows after grouping -> skipped.")
    else:
        print("LPIPS over-epochs: could not infer 'epoch' -> skipping line plot.")

    vals = pd.to_numeric(df[lp_col], errors="coerce").dropna().values
    if len(vals):
        plt.figure(figsize=(7,5))
        plt.hist(vals, bins=30)
        plt.xlabel("LPIPS"); plt.ylabel("Count")
        plt.title("Distribution of LPIPS distances (all prompts/pairs)")
        savefig(os.path.join(FIG_DIR, "fig_lpips_histogram.png"))
    else:
        print("LPIPS histogram: no numeric values -> skipped.")

df_lpips = load_lpips_table()
plot_lpips(df_lpips)
print("Done. Figures folder:", FIG_DIR)


Using LPIPS CSV: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/logs/eval_lpips_diversity.csv
LPIPS columns: ['epoch', 'lpips_within_mean', 'ema_used', 'trunc', 'samples_per_prompt', 'n_prompts']
LPIPS: no recognizable column -> skipped.
Done. Figures folder: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs


In [41]:
# plots
import os, glob, re, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------- CONFIG PATHS ---------
BASE    = "/content/drive/MyDrive/DCGAN-Text-to-Image"
LOG_DIR = os.path.join(BASE, "output_clip", "logs")
FIG_DIR = os.path.join(BASE, "output_clip", "figs")
os.makedirs(FIG_DIR, exist_ok=True)

TRAIN_CSV_GLOB        = os.path.join(LOG_DIR, "train_*.csv")
CLIP_SAMPLES_CSV_PATH = os.path.join(LOG_DIR, "eval_clip_samples_all.csv")
CLIP_SUMMARY_CSV_PATH = os.path.join(LOG_DIR, "eval_clip_summary.csv")  # optional; not required
LPIPS_CSV_PATH        = os.path.join(LOG_DIR, "eval_lpips_diversity.csv")
LPIPS_XLSX_PATH       = os.path.join(LOG_DIR, "eval_lpips_diversity.xlsx")
FID_TXT_GLOB          = os.path.join(LOG_DIR, "fid", "fid_epoch*.txt")

# --------- HELPERS ---------
def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    print("Saved:", path)

def latest_match(glob_pattern):
    files = sorted(glob.glob(glob_pattern))
    return files[-1] if files else None

def pick_col(df, candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the columns found: {candidates}")
    return None

def ensure_epoch_column(df, ckpt_col_candidates=("ckpt","checkpoint","gen_ckpt","file","path","checkpoint_path","model","src")):
    if "epoch" in df.columns:
        return df
    for c in ckpt_col_candidates:
        if c in df.columns:
            def parse_ep(x):
                m = re.search(r'gen[_\-]?(\d{1,5})\.pth', str(x))
                return int(m.group(1)) if m else None
            dfx = df.copy()
            dfx["epoch"] = dfx[c].map(parse_ep)
            return dfx
    return df

# --------- 1) TRAINING CURVES ---------
train_csv = latest_match(TRAIN_CSV_GLOB)
if train_csv and os.path.isfile(train_csv):
    print("Training CSV:", train_csv)
    tr = pd.read_csv(train_csv)
    if "epoch" in tr.columns:
        plt.figure(figsize=(9,5))
        if "d_loss" in tr.columns:
            plt.plot(tr["epoch"], tr["d_loss"], label="D loss")
        if "g_adv" in tr.columns:
            plt.plot(tr["epoch"], tr["g_adv"], label="G adv loss")
        # CLIP term
        clip_col = pick_col(tr, ["clip_loss","c_loss"])
        if clip_col:
            plt.plot(tr["epoch"], tr[clip_col], label="CLIP loss")
        gtot_col = pick_col(tr, ["g_total","g_tot","g_loss"])
        if gtot_col:
            plt.plot(tr["epoch"], tr[gtot_col], label="G total")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training losses over epochs")
        plt.legend()
        savefig(os.path.join(FIG_DIR, "fig_training_losses.png"))
    else:
        print("Training curves skipped: no 'epoch' in training CSV.")
else:
    print("Training curves skipped: no training CSV found.")

# --------- 2) CLIP PLOTS  ---------
if os.path.isfile(CLIP_SAMPLES_CSV_PATH):
    print("CLIP samples CSV:", CLIP_SAMPLES_CSV_PATH)
    cs = pd.read_csv(CLIP_SAMPLES_CSV_PATH)
    cs = ensure_epoch_column(cs)
    clip_col = pick_col(cs, ["clip_score","clip","similarity","score"], required=True)

    # 2a) CLIP over epochs  if epoch present
    if "epoch" in cs.columns and not cs["epoch"].isna().all():
        grp = (cs.dropna(subset=[clip_col,"epoch"])
                 .groupby("epoch")[clip_col]
                 .agg(mean="mean", std="std", n="count")
                 .reset_index())
        grp["ci_low"]  = grp["mean"] - 1.96*(grp["std"] / np.sqrt(grp["n"].clip(lower=1)))
        grp["ci_high"] = grp["mean"] + 1.96*(grp["std"] / np.sqrt(grp["n"].clip(lower=1)))
        grp = grp.sort_values("epoch")
        plt.figure(figsize=(8,5))
        plt.plot(grp["epoch"], grp["mean"], marker="o")
        plt.fill_between(grp["epoch"], grp["ci_low"], grp["ci_high"], alpha=0.2)
        plt.xlabel("Epoch"); plt.ylabel("CLIP similarity (higher is better)")
        plt.title("CLIP alignment over epochs")
        savefig(os.path.join(FIG_DIR, "fig_clip_over_epochs.png"))
    else:
        print("CLIP over-epochs skipped: could not infer 'epoch'.")

    # 2b) Histogram of all CLIP scores
    vals = pd.to_numeric(cs[clip_col], errors="coerce").dropna().values
    if len(vals):
        plt.figure(figsize=(7,5))
        plt.hist(vals, bins=30)
        plt.xlabel("CLIP similarity"); plt.ylabel("Count")
        plt.title("Distribution of CLIP similarities (all prompts/epochs)")
        savefig(os.path.join(FIG_DIR, "fig_clip_histogram.png"))
    else:
        print("CLIP histogram skipped: no numeric clip values.")

    # 2c) Per-prompt boxplot
    if "prompt" in cs.columns:
        prompts = list(dict.fromkeys(cs["prompt"].astype(str).tolist()))
        data = [pd.to_numeric(cs.loc[cs["prompt"]==p, clip_col], errors="coerce").dropna().values for p in prompts]
        data = [d for d in data if len(d)]
        labels = [p for p, d in zip(prompts, data) if len(d)]
        if data:
            plt.figure(figsize=(max(8, len(labels)*0.6), 5))
            plt.boxplot(data, tick_labels=labels, showfliers=False)
            plt.xticks(rotation=35, ha="right")
            plt.ylabel("CLIP similarity")
            plt.title("CLIP per prompt")
            savefig(os.path.join(FIG_DIR, "fig_clip_per_prompt_boxplot.png"))
        else:
            print("CLIP per-prompt boxplot skipped: no data after filtering.")
    else:
        print("CLIP per-prompt boxplot skipped: 'prompt' column missing.")
else:
    print("CLIP plots skipped: samples CSV not found.")

# --------- 3) LPIPS PLOTS  ---------
def load_lpips_df():
    if os.path.isfile(LPIPS_CSV_PATH):
        print("LPIPS CSV:", LPIPS_CSV_PATH)
        return pd.read_csv(LPIPS_CSV_PATH)
    if os.path.isfile(LPIPS_XLSX_PATH):
        print("Converting LPIPS XLSX -> CSV:", LPIPS_XLSX_PATH)
        df = pd.read_excel(LPIPS_XLSX_PATH, engine="openpyxl")
        df.to_csv(LPIPS_CSV_PATH, index=False)
        return df
    # try any match
    xlsx_any = latest_match(os.path.join(LOG_DIR, "**", "*lpips*.xlsx"))
    if xlsx_any and os.path.isfile(xlsx_any):
        print("Converting LPIPS XLSX -> CSV:", xlsx_any)
        df = pd.read_excel(xlsx_any, engine="openpyxl")
        df.to_csv(LPIPS_CSV_PATH, index=False)
        return df
    csv_any = latest_match(os.path.join(LOG_DIR, "**", "*lpips*.csv"))
    if csv_any and os.path.isfile(csv_any):
        print("LPIPS CSV:", csv_any)
        return pd.read_csv(csv_any)
    return None

lp = load_lpips_df()
if lp is None or len(lp)==0:
    print("LPIPS plots skipped: no LPIPS table found.")
else:
    print("LPIPS columns:", list(lp.columns))
    # Over epochs
    if "epoch" in lp.columns and "lpips_within_mean" in lp.columns:
        dplot = lp.dropna(subset=["epoch","lpips_within_mean"]).copy()
        dplot = dplot.sort_values("epoch")
        plt.figure(figsize=(8,5))
        plt.plot(dplot["epoch"], dplot["lpips_within_mean"], marker="o")
        plt.xlabel("Epoch"); plt.ylabel("LPIPS (within-prompt mean)")
        plt.title("LPIPS diversity over epochs (lower suggests less diversity)")
        savefig(os.path.join(FIG_DIR, "fig_lpips_over_epochs.png"))

        # Histogram
        vals = pd.to_numeric(dplot["lpips_within_mean"], errors="coerce").dropna().values
        if len(vals):
            plt.figure(figsize=(7,5))
            plt.hist(vals, bins=20)
            plt.xlabel("LPIPS within-prompt mean"); plt.ylabel("Count")
            plt.title("Distribution of LPIPS means across epochs")
            savefig(os.path.join(FIG_DIR, "fig_lpips_histogram.png"))
        else:
            print("LPIPS histogram skipped: no numeric values.")
    else:
        print("LPIPS plots skipped: expected columns 'epoch' and 'lpips_within_mean' not found.")

# --------- 4) FID BAR PLOT -------
fid_txts = sorted(glob.glob(FID_TXT_GLOB))
if fid_txts:
    rows = []
    for p in fid_txts:
        m = re.search(r'fid_epoch(\d+)\.txt', os.path.basename(p))
        ep = int(m.group(1)) if m else None
        try:
            with open(p, "r") as f:
                val = float(f.read().strip())
            rows.append((ep, val, p))
        except:
            pass
    if rows:
        rows = [(e,v,p) for (e,v,p) in rows if e is not None]
        rows.sort(key=lambda x: x[0])
        eps = [r[0] for r in rows]
        vals = [r[1] for r in rows]
        plt.figure(figsize=(9,5))
        plt.bar(eps, vals)
        plt.xlabel("Epoch"); plt.ylabel("FID (lower is better)")
        plt.title("FID by epoch")
        savefig(os.path.join(FIG_DIR, "fig_fid_bar.png"))
    else:
        print("FID plot skipped: no parsable FID rows.")
else:
    print("FID plot skipped: no fid_epoch*.txt files found.")

print("\nAll done. Figures in:", FIG_DIR)


Training CSV: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/logs/train_20251113_020008.csv
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_training_losses.png
CLIP samples CSV: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/logs/eval_clip_samples_all.csv
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_clip_over_epochs.png
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_clip_histogram.png
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_clip_per_prompt_boxplot.png
LPIPS CSV: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/logs/eval_lpips_diversity.csv
LPIPS columns: ['epoch', 'lpips_within_mean', 'ema_used', 'trunc', 'samples_per_prompt', 'n_prompts']
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_lpips_over_epochs.png
Saved: /content/drive/MyDrive/DCGAN-Text-to-Image/output_clip/figs/fig_lpips_histogram.png
Saved: /content/drive/MyDrive/DCGAN-Text-t